<a href="https://colab.research.google.com/github/DeepakSaini01/ML-WEEK-1/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:

!git clone https://github.com/DeepakSaini01/ML-WEEK-1.git

Cloning into 'ML-WEEK-1'...
remote: Enumerating objects: 134, done.
remote: Counting objects: 100% (134/134), done.
remote: Compressing objects: 100% (88/88), done.
remote: Total 134 (delta 45), reused 98 (delta 30), pack-reused 0 (from 0)
Receiving objects: 100% (134/134), 1.83 MiB | 8.21 MiB/s, done.
Resolving deltas: 100% (45/45), done.


# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DeepakSaini01/ML-WEEK-1/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

Unit of Analysis (Grain): One row represents one unique search query for a specific page URL (url + query) aggregated over a period.

Time Window: Observed data spans from the earliest recorded date (min_date) to the latest recorded date (max_date) present in content_refresh_anonymized.csv.

In [18]:
import os
import pandas as pd

# Load dataset using cloned repository path
file_path = 'ML-WEEK-1/data/raw/content_refresh_anonymized.csv'
if not os.path.exists(file_path):
    file_path = 'data/raw/content_refresh_anonymized.csv'

df = pd.read_csv(file_path)

# Verify time window
date_cols = [c for c in df.columns if 'date' in c.lower()]
if date_cols:
    date_col = date_cols[0]
    df[date_col] = pd.to_datetime(df[date_col])
    print(f"Time Window: {df[date_col].min()} to {df[date_col].max()}")
else:
    print("Time Window: No explicit date column present in raw CSV schema.")

# Verify grain uniqueness (URL + Query)
if 'url' in df.columns and 'query' in df.columns:
    total_rows = len(df)
    unique_pairs = len(df.groupby(['url', 'query']))
    print(f"Total Rows: {total_rows}")
    print(f"Unique (url, query) pairs: {unique_pairs}")
    print(f"Is grain strictly unique per row? {total_rows == unique_pairs}")

if 'days_since_last_update' in df.columns:
    df['days_since_last_update'] = pd.to_datetime(df['days_since_last_update'])
    print(f"Last Update Range: {df['days_since_last_update'].min()} to {df['days_since_last_update'].max()}")
elif 'content_age_days' in df.columns:
    print(f"Content Age Range (Days): {df['content_age_days'].min()} to {df['content_age_days'].max()}")

Time Window: 1970-01-01 00:00:00.000000001 to 1970-01-01 00:00:00.000000373
Last Update Range: 1970-01-01 00:00:00.000000001 to 1970-01-01 00:00:00.000000373


## 2. Fields: feature / label / context / excluded

Categorization:

Features: impressions (total search views), clicks (total user clicks), position (average search rank position).

Label: ctr (Click-Through Rate, calculated as clicks / impressions).

Context: url (page identifier), query (search key phrase), date (record timestamp).

Excluded: raw_id / internal system flags — Reason for exclusion: Metadata irrelevant to search ranking performance that risks introducing noise.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Check schema and field null counts
print("Dataset Columns:", list(df.columns))
print("\nDataset Info:")
print(df.info())

print("\nMissing values per field:")
print(df.isnull().sum())

Dataset Columns: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']

Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 44 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                 

## 3. Verify it with queries (grain, counts, missing values, windows)

Contract Verification Claims:

Total row counts and shape confirm expected dataset dimensions.

Primary numeric features (impressions, clicks, position) contain non-null, valid numeric ranges.

Verified zero duplicate entries exist across the dataset.

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Verification checks
print(f"Total Dataset Shape: {df.shape}")

numeric_cols = df.select_dtypes(include=['number']).columns.tolist()
print("\nSummary Statistics for Numeric Columns:")
print(df[numeric_cols].describe())

duplicates = df.duplicated().sum()
print(f"\nDuplicate Rows Count: {duplicates}")

Total Dataset Shape: (30000, 44)

Summary Statistics for Numeric Columns:
       search_volume   competition           cpc    word_count     char_count  \
count   27532.000000  27532.000000  27532.000000  22301.000000   22301.000000   
mean      158.882391      0.146954      0.485342   3107.760325   20665.277835   
std      1518.270825      0.285241      2.101560   1452.382598   10115.344042   
min         0.000000      0.000000      0.000000      8.000000      40.000000   
25%         0.000000      0.000000      0.000000   2413.000000   15644.000000   
50%        10.000000      0.000000      0.000000   2877.000000   19116.000000   
75%        20.000000      0.130000      0.000000   3666.000000   24011.000000   
max     74000.000000      1.000000    100.360000   9546.000000  111158.000000   

       impressions_90d    clicks_90d  pageviews_90d  sessions_90d  \
count     30000.000000  30000.000000   30000.000000  30000.000000   
mean       5200.366300     16.097333      49.942467     37

## 4. Data limits

Known Limitations:

Zero-Impression Truncation: Search queries resulting in zero impressions are omitted entirely from Search Console exports.

Anonymization Scope: Anonymized identifiers mask specific domain context, restricting domain-level enrichment.

Time Window Boundary: Model assumptions apply strictly to the observed timeframe and cannot account for seasonal search volatility outside this window.

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Check feature distributions and zero-value constraints
for col in numeric_cols:
    zero_count = (df[col] == 0).sum()
    print(f"Column '{col}' — Zero values count: {zero_count} ({(zero_count / len(df)) * 100:.2f}%)")

Column 'search_volume' — Zero values count: 11081 (36.94%)
Column 'competition' — Zero values count: 16379 (54.60%)
Column 'cpc' — Zero values count: 20679 (68.93%)
Column 'word_count' — Zero values count: 0 (0.00%)
Column 'char_count' — Zero values count: 0 (0.00%)
Column 'impressions_90d' — Zero values count: 0 (0.00%)
Column 'clicks_90d' — Zero values count: 13204 (44.01%)
Column 'pageviews_90d' — Zero values count: 125 (0.42%)
Column 'sessions_90d' — Zero values count: 0 (0.00%)
Column 'users_90d' — Zero values count: 0 (0.00%)
Column 'engaged_sessions_90d' — Zero values count: 21629 (72.10%)
Column 'ai_sessions_90d' — Zero values count: 28070 (93.57%)
Column 'scroll_events_90d' — Zero values count: 11235 (37.45%)
Column 'days_with_impressions' — Zero values count: 0 (0.00%)
Column 'days_with_sessions' — Zero values count: 0 (0.00%)
Column 'impressions_last_30d' — Zero values count: 2547 (8.49%)
Column 'clicks_last_30d' — Zero values count: 18365 (61.22%)
Column 'sessions_last_30d'

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.